In [1]:
from datasets import load_dataset

ag = load_dataset("ag_news")
print(ag)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [2]:
print(ag["train"].column_names)

['text', 'label']


In [3]:
print(ag["train"][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [4]:
dbpedia = load_dataset("dbpedia_14")
print(dbpedia)

README.md: 0.00B [00:00, ?B/s]

dbpedia_14/train-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

dbpedia_14/test-00000-of-00001.parquet:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 70000
    })
})


In [5]:
yahoo = load_dataset("yahoo_answers_topics")
print(yahoo)

README.md: 0.00B [00:00, ?B/s]

yahoo_answers_topics/train-00000-of-0000(…):   0%|          | 0.00/241M [00:00<?, ?B/s]

yahoo_answers_topics/train-00001-of-0000(…):   0%|          | 0.00/270M [00:00<?, ?B/s]

yahoo_answers_topics/test-00000-of-00001(…):   0%|          | 0.00/21.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1400000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/60000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'topic', 'question_title', 'question_content', 'best_answer'],
        num_rows: 1400000
    })
    test: Dataset({
        features: ['id', 'topic', 'question_title', 'question_content', 'best_answer'],
        num_rows: 60000
    })
})


In [6]:
ag = load_dataset("ag_news")
dbpedia = load_dataset("dbpedia_14")
yahoo = load_dataset("yahoo_answers_topics")

In [7]:
def describe(ds, name):
    print(f"\n=== {name} ===")
    print(ds)
    for split in ds.keys():
        print(f"{split}: {len(ds[split])} rows | columns: {ds[split].column_names}")

describe(ag, "AG News")
describe(dbpedia, "DBpedia 14")
describe(yahoo, "Yahoo Answers Topics")


=== AG News ===
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
train: 120000 rows | columns: ['text', 'label']
test: 7600 rows | columns: ['text', 'label']

=== DBpedia 14 ===
DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 70000
    })
})
train: 560000 rows | columns: ['label', 'title', 'content']
test: 70000 rows | columns: ['label', 'title', 'content']

=== Yahoo Answers Topics ===
DatasetDict({
    train: Dataset({
        features: ['id', 'topic', 'question_title', 'question_content', 'best_answer'],
        num_rows: 1400000
    })
    test: Dataset({
        features: ['id', 'topic', 'question_title', 'question_content', 'best_answer'],
        num_rows: 60000
    })
})
train: 1400

In [8]:
import pandas as pd

ag_test = pd.DataFrame(ag["test"])[["text", "label"]]
ag_test.to_csv("agnews_test_en.csv", index=False)

ag_pool = pd.DataFrame(ag["train"].shuffle(seed=42).select(range(10000)))[["text", "label"]]
ag_pool.to_csv("agnews_train_pool_en_10k.csv", index=False)

print("AG saved:", ag_test.shape, ag_pool.shape)

AG saved: (7600, 2) (10000, 2)


In [9]:
def dbpedia_to_df(split):
    df = pd.DataFrame(dbpedia[split])[["title", "content", "label"]]
    df["text"] = df["title"].fillna("").str.strip() + ". " + df["content"].fillna("").str.strip()
    df["text"] = df["text"].str.strip()
    return df[["text", "label"]]

db_test = dbpedia_to_df("test")
db_test.to_csv("dbpedia14_test_en.csv", index=False)

db_pool = dbpedia_to_df("train").sample(n=10000, random_state=42)
db_pool.to_csv("dbpedia14_train_pool_en_10k.csv", index=False)

print("DBpedia saved:", db_test.shape, db_pool.shape)

DBpedia saved: (70000, 2) (10000, 2)


In [11]:
def yahoo_to_df(split, max_chars=1200):
    df = pd.DataFrame(
        yahoo[split]
    )[["question_title", "question_content", "best_answer", "topic"]].copy()

    df["text"] = (
        "Title: " + df["question_title"].fillna("").str.strip()
        + "\nQuestion: " + df["question_content"].fillna("").str.strip()
        + "\nAnswer: " + df["best_answer"].fillna("").str.strip()
    ).str.strip()

    # truncate to control token cost
    df["text"] = df["text"].str.slice(0, max_chars)

    # rename topic -> label for consistency across datasets
    df = df.rename(columns={"topic": "label"})

    return df[["text", "label"]]

yh_test = yahoo_to_df("test")
yh_test.to_csv("yahoo_topics_test_en.csv", index=False)

yh_pool = yahoo_to_df("train").sample(n=10000, random_state=42)
yh_pool.to_csv("yahoo_topics_train_pool_en_10k.csv", index=False)

print("Yahoo saved:", yh_test.shape, yh_pool.shape)
print("Unique labels:", sorted(yh_test["label"].unique())[:15], " ... total:", yh_test["label"].nunique())

Yahoo saved: (60000, 2) (10000, 2)
Unique labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  ... total: 10


In [12]:
yh_pool.head()

,text,label
915087,Title: I'm expecting a marriage proposal when?...,7
891768,Title: what is the preferred process of recrui...,6
131043,Title: history about lounge cancer?\nQuestion:...,3
220143,Title: girls -- wanting sex and ovualation/per...,2
898067,Title: How can i contact Stephen colbert?\nQue...,7
